# Apache Hudi Core Conceptions (4) - Clustering

## 1. Configuration

In [1]:
%%configure -f
{
    "conf" : {
        "spark.jars":"hdfs:///tmp/hudi-spark-bundle.jar",            
        "spark.serializer":"org.apache.spark.serializer.KryoSerializer",
        "spark.sql.extensions":"org.apache.spark.sql.hudi.HoodieSparkSessionExtension",
        "spark.sql.catalog.spark_catalog":"org.apache.spark.sql.hudi.catalog.HoodieCatalog",
        "spark.driver.memory":"4g",
        "spark.executor.core":"1",
        "spark.executor.memory":"4g"
    }
}

In [2]:
%%sh
# deploy hudi bundle jar
hdfs dfs -copyFromLocal -f /usr/lib/hudi/hudi-spark-bundle.jar /tmp/hudi-spark-bundle.jar
hdfs dfs -ls /tmp/hudi-spark-bundle.jar
# deploy hudi-stat.sh - a utility shell script 
wget https://github.com/bluishglc/hudi-core-conceptions/releases/download/v1.0/hudi-stat.sh -O ~/hudi-stat.sh &>/dev/null
chmod a+x ~/hudi-stat.sh
ls ~/hudi-stat.sh

-rw-r--r--   1 emr-notebook hdfsadmingroup   61421977 2023-03-06 12:16 /tmp/hudi-spark-bundle.jar
/home/emr-notebook/hudi-stat.sh


In [3]:
%%html
<style>
table {float:left}
</style>

## 2. Test Case 1 - Sync Clustering ( Inline Schedule + Inline Execute )

### 2.1. Test Plan

Step No.|Action|Volume / Partition |File System
:--------:|:------|:------|:----------
1|Insert|15MB|1 Small FileGroup
2|Insert|341MB|3 Big FileGroups
3|Insert|377MB|3 Big FileGroups + 1 Small FileGroup
4|Insert||

### 2.2. Key Settings

KEY|DEFAULT VALUE|SET VALUE
:---|:---|:---
hoodie.clustering.inline|false|true
hoodie.clustering.inline.max.commits|4|2
hoodie.clustering.async.enabled|false|false
hoodie.clustering.plan.strategy.target.file.max.bytes|1073741824 / 1GB|419430400 / 400MB
hoodie.clustering.plan.strategy.small.file.limit|314572800 / 300MB|209715200 / 200MB
hoodie.clustering.plan.strategy.sort.columns|-|review_date
hoodie.parquet.small.file.limit|104857600 / 100MB | 0
hoodie.copyonwrite.record.size.estimate|1024|175

### 2.3. Set Variables

In [4]:
%%sql
set TABLE_NAME=reviews_cow_clustering_1

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
4,application_1678096020253_0011,spark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [5]:
%env TABLE_NAME=reviews_cow_clustering_1

env: TABLE_NAME=reviews_cow_clustering_1


In [6]:
%%sql
set TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_cow_clustering_1

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [7]:
%env TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_cow_clustering_1

env: TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_cow_clustering_1


### 2.4. Create Table

In [8]:
%%sh
echo $(basename $TABLE_PATH)
aws s3 rm $TABLE_PATH --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 5

reviews_cow_clustering_1


In [9]:
%%sql
drop table if exists ${TABLE_NAME}

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [10]:
%%sql
create table if not exists ${TABLE_NAME}(
    review_id string, 
    star_rating int, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'cow',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.clustering.inline = 'true',
    -- it NOT allowed to set inline & schelue.inline both true.
    -- hoodie.clustering.schedule.inline = 'true',
    hoodie.clustering.inline.max.commits = '2',
    hoodie.clustering.async.enabled = 'false',
    hoodie.clustering.plan.strategy.small.file.limit = '209715200',
    hoodie.clustering.plan.strategy.target.file.max.bytes = '419430400',
    hoodie.clustering.plan.strategy.sort.columns = 'review_date',
    hoodie.parquet.small.file.limit = '0',
    hoodie.copyonwrite.record.size.estimate = '175'
);

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

### 2.5. Insert 15MB ( + 1 Small FileGroup )

In [11]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1998;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Output()

In [12]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage


[ TIMELINE ]

╔═════╤═══════════════════╤════════╤═══════════╤═════════════╤═════════════╤═════════════╗
║ No. │ Instant           │ Action │ State     │ Requested   │ Inflight    │ Completed   ║
║     │                   │        │           │ Time        │ Time        │ Time        ║
╠═════╪═══════════════════╪════════╪═══════════╪═════════════╪═════════════╪═════════════╣
║ 0   │ 20230306121756218 │ commit │ COMPLETED │ 03-06 12:17 │ 03-06 12:18 │ 03-06 12:18 ║
╚═════╧═══════════════════╧════════╧═══════════╧═════════════╧═════════════╧═════════════╝

[ COMMITS ]

╔═══════════════════╤═════════════════════╤═══════════════════╤═════════════════════╤══════════════════════════╤═══════════════════════╤══════════════════════════════╤══════════════╗
║ CommitTime        │ Total Bytes Written │ Total Files Added │ Total Files Updated │ Total Partitions Written │ Total Records Written │ Total Update Records Written │ Total Errors ║
╠═══════════════════╪═════════════════════╪════════════════

### 2.6. Insert 341MB ( + 3 Big FileGroups )

In [ ]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2010;

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

## 3. Test Case 2 - Semi-Sync Clustering ( Inline Schedule + Offline Execute )

### 3.1. Test Plan

Step No.|Action|Volume / Partition |File System
:--------:|:------|:------|:----------
1|Insert|15MB|1 Small FileGroup
2|Insert|341MB|3 Big FileGroups
3|Insert|377MB|3 Big FileGroups + 1 Small FileGroup
4|Insert||

### 3.2. Key Settings

KEY|DEFAULT VALUE|SET VALUE
:---|:---|:---
hoodie.clustering.inline|false|true
hoodie.clustering.inline.max.commits|4|2
hoodie.clustering.async.enabled|false|false
hoodie.clustering.plan.strategy.target.file.max.bytes|1073741824 / 1GB|419430400 / 400MB
hoodie.clustering.plan.strategy.small.file.limit|314572800 / 300MB|209715200 / 200MB
hoodie.clustering.plan.strategy.sort.columns|-|review_date
hoodie.parquet.small.file.limit|104857600 / 100MB | 0
hoodie.copyonwrite.record.size.estimate|1024|175

### 3.3. Set Variables

In [ ]:
%%sql
set TABLE_NAME=reviews_cow_clustering_2

In [ ]:
%env TABLE_NAME=reviews_cow_clustering_2

In [ ]:
%%sql
set TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_cow_clustering_2

In [ ]:
%env TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_cow_clustering_2

### 3.4. Create Table

In [ ]:
%%sh
echo $(basename $TABLE_PATH)
aws s3 rm $TABLE_PATH --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 5

In [ ]:
%%sql
drop table if exists ${TABLE_NAME}

In [ ]:
%%sql
create table if not exists ${TABLE_NAME}(
    review_id string, 
    star_rating int, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'cow',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.clustering.inline = 'false',
    hoodie.clustering.schedule.inline = 'true',
    hoodie.clustering.async.enabled = 'false',
    hoodie.clustering.inline.max.commits = '2',
    hoodie.clustering.plan.strategy.small.file.limit = '209715200',
    hoodie.clustering.plan.strategy.target.file.max.bytes = '419430400',
    hoodie.clustering.plan.strategy.sort.columns = 'review_date',
    hoodie.parquet.small.file.limit = '0',
    hoodie.copyonwrite.record.size.estimate = '175'
);

### 3.5. Insert 15MB ( + 1 Small FileGroup )

In [ ]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1998;

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

### 3.6. Insert 341MB ( + 3 Big FileGroups )

In [ ]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2010;

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

### 3.7. Offline Eexecute 354MB ( ∑ 1 Clustered FileGroup )

In [ ]:
%%sh
# it's required for current user (emr-notebook) to get sudo permission
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class 'org.apache.hudi.utilities.HoodieClusteringJob' \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'execute' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" \
  --hoodie-conf "hoodie.clustering.async.enabled=true" \
  --hoodie-conf "hoodie.clustering.async.max.commits=2" \
  --hoodie-conf "hoodie.clustering.plan.strategy.small.file.limit=209715200" \
  --hoodie-conf "hoodie.clustering.plan.strategy.target.file.max.bytes=419430400" \
  --hoodie-conf "hoodie.clustering.plan.strategy.sort.columns=review_date" > ~/${TABLE_NAME}.clustering.execute.out &>/dev/null

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

## 4. Test Case 3 - Async Clustering ( Offline Schedule + Offline Execute )

### 4.1. Test Plan

Step No.|Action|Volume / Partition |File System
:--------:|:------|:------|:----------
1|Insert|15MB|1 Small FileGroup
2|Insert|341MB|3 Big FileGroups
3|Async Clustering Schedule|-|-
4|Async Clustering Eexecute|354MB|∑ 1 Clustered FileGroup
5|Insert|377MB|3 Big FileGroups + 1 Small FileGroup
6|Insert||

### 4.2. Key Settings

KEY|DEFAULT VALUE|SET VALUE
:---|:---|:---
hoodie.clustering.async.enabled|false|true
hoodie.clustering.async.max.commits|4|2
hoodie.clustering.plan.strategy.target.file.max.bytes|1073741824 / 1GB|419430400 / 400MB
hoodie.clustering.plan.strategy.small.file.limit|314572800 / 300MB|209715200 / 200MB
hoodie.clustering.plan.strategy.sort.columns|-|review_date
hoodie.parquet.small.file.limit|104857600 / 100MB | 0
hoodie.copyonwrite.record.size.estimate|1024|175

### 4.3. Set Variables

In [ ]:
%%sql
set TABLE_NAME=reviews_cow_clustering_3

In [ ]:
%env TABLE_NAME=reviews_cow_clustering_3

In [ ]:
%%sql
set TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_cow_clustering_3

In [ ]:
%env TABLE_PATH=s3://glc-examples/hudi-core-conceptions/reviews_cow_clustering_3

### 4.4. Create Table

In [ ]:
%%sh
echo $(basename $TABLE_PATH)
aws s3 rm $TABLE_PATH --recursive &>/dev/null
rm -rf ~/${TABLE_NAME}
sleep 5

In [ ]:
%%sql
drop table if exists ${TABLE_NAME}

In [ ]:
%%sql
create table if not exists ${TABLE_NAME}(
    review_id string, 
    star_rating int, 
    review_body string, 
    review_date date, 
    year long,
    timestamp long,
    parity int
)
using hudi
location '${TABLE_PATH}'
partitioned by (parity)
options ( 
    type = 'cow',  
    primaryKey = 'review_id', 
    preCombineField = 'timestamp',
    hoodie.clustering.inline = 'false',
    hoodie.clustering.schedule.inline = 'false',
    hoodie.clustering.async.enabled = 'true',
    hoodie.clustering.async.max.commits = '2',
    hoodie.clustering.plan.strategy.small.file.limit = '209715200',
    hoodie.clustering.plan.strategy.target.file.max.bytes = '419430400',
    hoodie.clustering.plan.strategy.sort.columns = 'review_date',
    hoodie.parquet.small.file.limit = '0',
    hoodie.copyonwrite.record.size.estimate = '175'
);

### 4.5. Insert 15MB ( + 1 Small FileGroup )

In [ ]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 1998;

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

### 4.6. Insert 341MB ( + 3 Max FileGroups )

In [ ]:
%%sql
insert into 
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year = 2010;

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

### 4.7. Async Clustering Schedule

In [ ]:
%%sh
# it's required for current user (emr-notebook) to get sudo permission
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class 'org.apache.hudi.utilities.HoodieClusteringJob' \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'schedule' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" \
  --hoodie-conf "hoodie.clustering.async.enabled=true" \
  --hoodie-conf "hoodie.clustering.async.max.commits=2" \
  --hoodie-conf "hoodie.clustering.plan.strategy.small.file.limit=209715200" \
  --hoodie-conf "hoodie.clustering.plan.strategy.target.file.max.bytes=419430400" \
  --hoodie-conf "hoodie.clustering.plan.strategy.sort.columns=review_date" > ~/${TABLE_NAME}.clustering.schedule.out &>/dev/null

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

### 4.8. Async Clustering Eexecute ( 354MB -> ∑ 1 Clustered FileGroup )

In [ ]:
%%sh
# it's required for current user (emr-notebook) to get sudo permission
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class 'org.apache.hudi.utilities.HoodieClusteringJob' \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'execute' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" \
  --hoodie-conf "hoodie.clustering.async.enabled=true" \
  --hoodie-conf "hoodie.clustering.async.max.commits=2" \
  --hoodie-conf "hoodie.clustering.plan.strategy.small.file.limit=209715200" \
  --hoodie-conf "hoodie.clustering.plan.strategy.target.file.max.bytes=419430400" \
  --hoodie-conf "hoodie.clustering.plan.strategy.sort.columns=review_date" > ~/${TABLE_NAME}.clustering.execute.out &>/dev/null

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

### 4.9. Insert ( 377MB -> 4 FileGroups )

In [ ]:
%%sql
insert into
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year in (2005, 2009);

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

### 4.10. Insert ( 301MB -> 3 FileGroups )

In [ ]:
%%sql
insert into
    ${TABLE_NAME}
select 
    review_id, 
    star_rating, 
    review_body, 
    review_date, 
    year,
    unix_timestamp(current_timestamp()) as timestamp,
    mod(crc32(review_id), 2) as parity
from
    reviews
where
    year in (2006, 2007);

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage

### 4.11. Async ( Schedule + Execute ) Clustering

In [ ]:
%%sh
# 1. it's required for current user (emr-notebook) to get sudo permission
# 2. commented-out hoodie-conf items are default settings
sudo -u hadoop spark-submit \
  --jars '/usr/lib/hudi/hudi-spark-bundle.jar' \
  --class 'org.apache.hudi.utilities.HoodieClusteringJob' \
  /usr/lib/hudi/hudi-utilities-bundle.jar \
  --spark-memory '4g' \
  --mode 'scheduleAndExecute' \
  --base-path "$TABLE_PATH" \
  --table-name "$TABLE_NAME" \
  --hoodie-conf "hoodie.clustering.async.enabled=true" \
  --hoodie-conf "hoodie.clustering.async.max.commits=2" \
  --hoodie-conf "hoodie.clustering.plan.strategy.small.file.limit=209715200" \
  --hoodie-conf "hoodie.clustering.plan.strategy.target.file.max.bytes=419430400" \
  --hoodie-conf "hoodie.clustering.plan.strategy.sort.columns=review_date" > ~/${TABLE_NAME}.clustering.scheduleAndExecute.out &>/dev/null

In [ ]:
%%sh
~/hudi-stat.sh $TABLE_PATH timeline commits storage